# Skrub API example

The purpose of this notebook is to reconstruct an [example from the skrub tutorial](https://skrub-data.org/stable/auto_examples/data_ops/1110_data_ops_intro.html)  using ICO framework.


## 1. Creating a data source for training data
Here the `@source` decorator is used to define  `load_train_salary_data` as a data source

So far, the most common scenario for ICO has been modeling data pipelines where data sample is an atomic element and a dataset is represented as `Iterator[I]`.

💡 But there is a key difference with sklearn - it accepts whole dataset at once (as a table) for fitting and transforming. 

In the current version of ICO, we still need to implement that interface and return an `Iterator[I]`


In [1]:
from collections.abc import Iterable
from dataclasses import dataclass

import pandas as pd
from skrub.datasets import fetch_employee_salaries

from ico import source


@dataclass(frozen=True, slots=True)
class XYData:
    x: pd.DataFrame
    y: pd.DataFrame


@source()
def load_train_salary_data() -> Iterable[XYData]:
    training_data: pd.DataFrame = fetch_employee_salaries(
        split="train"
    ).employee_salaries  # type: ignore

    yield XYData(
        x=training_data.drop(columns="current_annual_salary"),  # type: ignore
        y=training_data["current_annual_salary"],  # type: ignore
    )

# 2. Building train data plan
- make use of `TableVectorizer` as an operator for fitting and transforming
- connect the data source and vectorization
- 💡 we need to use `.stream()` to make `vectorize_fit_transform` iterable

In [2]:
from skrub import TableVectorizer

from ico import operator

vectorizer = TableVectorizer()


@operator()
def vectorize_fit_transform(input: XYData) -> XYData:
    return XYData(
        x=vectorizer.fit_transform(input.x),  # type: ignore
        y=input.y,
    )


train_data_plan = load_train_salary_data | vectorize_fit_transform.stream()
train_data_plan.name = "Train Data Plan"

### 2.1 Visualization
Any pipeline can be visualized. In the example above, you can see what is going on in the training data plan:
- the source outputs `XYData`
- the stream iterate over it and applies `vectorize_fit_transform`, then yields the result

In [3]:
train_data_plan.describe()

─────────────────────────────────────────── Flow plan: Train Data Plan ────────────────────────────────────────────

 Flow                                 Signature                            Name 
 📚IcoSource(load_train_salary_data)  () → Iterator[XYData]                     
 ╭── for each in 🎞️ IcoStream()        Iterator[XYData] → Iterator[XYData]       
 │   vectorize_fit_transform          XYData → XYData                           
 ╰─▸ yield                            Iterator[XYData]

## 3. Train plan: fitting the regressor

- 💡 Key concept here: we interpret `HistGradientBoostingRegressor` as a Context and transform it with `IcoContextOperator`
- `IcoContextOperator` takes Input and Context, and returns Context' (e.g. modified context)
- `IcoEpoch` applies a context operator to each data item in a source. This is a learning template implemented in this class.
- As you may notice, `IcoEpoch` is used as a corner case here, because sklearn operate on tables (e.g. datasets), but ICO in its current version has an abstraction for `Iterable`. 

In [4]:
from sklearn.ensemble import HistGradientBoostingRegressor

from apriori.ico.core.epoch import IcoEpoch
from ico import context_operator

RegressorType = HistGradientBoostingRegressor


@context_operator()
def fit_regressor(item: XYData, context: RegressorType) -> RegressorType:
    return context.fit(item.x, item.y)  # type: ignore


train_plan = IcoEpoch(source=train_data_plan, context_operator=fit_regressor)

`.describe()` now shows the complete training plan. `IcoEpoch` has special renderer to display the logic behind it.

In [5]:
train_plan.describe()

───────────────────────────────────────────────── Flow plan: None ─────────────────────────────────────────────────

 Flow                                 Signature                                                                    
 🧠IcoEpoch()                         Iterator[XYData], HistGradientBoostingRegressor → HistGradientBoostingRegr…  
 ╭── for each in                      Iterator[XYData]                                                             
 │   📚IcoSource(load_train_salary_…  () → Iterator[XYData]                                                        
 │   ╭── for each in 🎞️ IcoStream()    Iterator[XYData] → Iterator[XYData]                                          
 │   │   vectorize_fit_transform      XYData → XYData                                                              
 │   ╰─▸ yield                        Iterator[XYData]                                                             
 ├─▸ apply                            XYData, HistGradientBoostingRegressor                                        
 │   fit_regressor                    XYData, HistGradientBoostingRegressor → HistGradientBoostingRegressor        
 ╰─▸ emit                             HistGradientBoostingRegressor

We can already use `train_plan' to fit a regressor. 

Next, we will build a validation plan and combine it with training plan.

In [6]:
regressor = HistGradientBoostingRegressor()
trained_regressor = train_plan(regressor)
trained_regressor

,loss,'squared_error'
,quantile,None
,learning_rate,0.1
,max_iter,100
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'


## 4. Validation plan
- for validation we use `split='test'`
- in vectorization we use `vectorizer.transform(input.x)`, because it has already been fit.
- 💡 a key aspect of sklearn is that it uses stateful operators. Like 'vectorizer' - it should be fit once and then used for transforming
- We use `.score' for given regressor as a metric and print it for demonstration purposes

In [7]:
@source()
def load_test_salary_data() -> Iterable[XYData]:
    test_data: pd.DataFrame = fetch_employee_salaries(split="test").employee_salaries  # type: ignore

    yield XYData(
        x=test_data.drop(columns="current_annual_salary"),  # type: ignore
        y=test_data["current_annual_salary"],  # type: ignore
    )


@operator()
def vectorize_transform(input: XYData) -> XYData:
    return XYData(
        x=vectorizer.transform(input.x),  # type: ignore
        y=input.y,
    )


@context_operator()
def validate_regressor(item: XYData, regressor: RegressorType) -> RegressorType:
    print(f"{regressor.score(item.x, item.y)=}")  # type: ignore
    return regressor


val_data_plan = load_test_salary_data | vectorize_transform.stream()
val_plan = IcoEpoch(source=val_data_plan, context_operator=validate_regressor)

val_plan.describe()

───────────────────────────────────────────────── Flow plan: None ─────────────────────────────────────────────────

 Flow                                Signature                                                                     
 🧠IcoEpoch()                        Iterator[XYData], HistGradientBoostingRegressor → HistGradientBoostingRegr…   
 ╭── for each in                     Iterator[XYData]                                                              
 │   📚IcoSource(load_test_salary_…  () → Iterator[XYData]                                                         
 │   ╭── for each in 🎞️ IcoStream()   Iterator[XYData] → Iterator[XYData]                                           
 │   │   vectorize_transform         XYData → XYData                                                               
 │   ╰─▸ yield                       Iterator[XYData]                                                              
 ├─▸ apply                           XYData, HistGradientBoostingRegressor                                         
 │   validate_regressor              XYData, HistGradientBoostingRegressor → HistGradientBoostingRegressor         
 ╰─▸ emit                            HistGradientBoostingRegressor

# 5. Combining things together

- `train_plan` and `val_plan` are safe to connect, because they both take and return HistGradientBoostingRegressor, so mypy will be happy.


In [8]:
plan = train_plan | val_plan
plan.describe()

───────────────────────────────────────── Flow plan: IcoEpoch | IcoEpoch ──────────────────────────────────────────

 Flow                                 Signature                                                                    
 🧠IcoEpoch()                         Iterator[XYData], HistGradientBoostingRegressor → HistGradientBoostingRegr…  
 ╭── for each in                      Iterator[XYData]                                                             
 │   📚IcoSource(load_train_salary_…  () → Iterator[XYData]                                                        
 │   ╭── for each in 🎞️ IcoStream()    Iterator[XYData] → Iterator[XYData]                                          
 │   │   vectorize_fit_transform      XYData → XYData                                                              
 │   ╰─▸ yield                        Iterator[XYData]                                                             
 ├─▸ apply                            XYData, HistGradientBoostingRegressor                                        
 │   fit_regressor                    XYData, HistGradientBoostingRegressor → HistGradientBoostingRegressor        
 ╰─▸ emit                             HistGradientBoostingRegressor                                                
 🧠IcoEpoch()                         Iterator[XYData], HistGradientBoostingRegressor → HistGradientBoostingRegr…  
 ╭── for each in                      Iterator[XYData]                                                             
 │   📚IcoSource(load_test_salary_d…  () → Iterator[XYData]                                                        
 │   ╭── for each in 🎞️ IcoStream()    Iterator[XYData] → Iterator[XYData]                                          
 │   │   vectorize_transform          XYData → XYData                                                              
 │   ╰─▸ yield                        Iterator[XYData]                                                             
 ├─▸ apply                            XYData, HistGradientBoostingRegressor                                        
 │   validate_regressor               XYData, HistGradientBoostingRegressor → HistGradientBoostingRegressor        
 ╰─▸ emit                             HistGradientBoostingRegressor

## 6. Executing the plan
- 💡 the plan is just a declaration of actions, siumilar to skrub
- to execute it, we need to call it explicitly

In [9]:
regressor = HistGradientBoostingRegressor()

trained_regressor = plan(regressor)

regressor.score(item.x, item.y)=0.9396885191113729


## 7. 💡 Conclusions
1. An example from skrub tutorial can be modeled in ICO
2. Both aproaches have declarative level for pipeline representation
3. Both aproaches have visualization feature
4. ICO was not intended in the first place to work with tabular data, but it abstraction model can handle it. Futher extensions could make it more suitable for such data pipelines. 
5. sklearn operators (e.g. estimators and transformers) are statefull. A pipeline may include many operators with state, whitch makes things more complicated.
6. ICO is disigned in a way to separate mutable context from data. So sklearn pipelines need to be studied futher to address this difference.
7. Skrub already have a solution for state management via pickle API. ICO has such feature in the roadmap and is should be possible to implement.
8. It seems that a special layer for sklearn can be built on top of ICO to implement something similar to the skrub idea, with type safety and other ICO features included.
